In [69]:
import pandas as pd

# Load saved training and test CSV files from week 3
train = pd.read_csv("data/cleaned_train.csv")
test = pd.read_csv("data/cleaned_test.csv")

print("Training shape: ", train.shape)
print("Test shape: ", test.shape)

Training shape:  (117604, 503)
Test shape:  (11973, 503)


In [70]:
target = "ClosePrice"

city_cols = [
    col for col in train.columns
    if col.startswith("City_grouped_")
]

postal_cols = [
    col for col in train.columns
    if col.startswith("PostalCode_grouped_")
]

county_cols = [
    col for col in train.columns 
    if col.startswith("CountyOrParish_")
]

features_sets = {
    "Basic": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet"
        ],

    "With Property Features": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN"
    ],

    "With Missing LotSize Flagged": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "Missing_LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN"
    ],

    "With Location Features (City Only)": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN",
    ] + city_cols,

    "With Location Features (PostalCode Only)": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN"
    ] + postal_cols,

    "With Location Features (County Only)": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN"
    ] + county_cols,

    "With All Location Features": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN"
    ] + city_cols + postal_cols + county_cols

}


In [71]:
for name, features, in features_sets.items():

    # Define features and target
    X_train = train[features]   # Training Set
    y_train = train[target]

    X_test = test[features]     # Test Set
    y_test = test[target]

    # Ensure no missing values
    print(f"\n{name}")
    print("----------------------")
    missing = X_train.columns[X_train.isna().any()]     # Training Set
    print(missing)
    for col in missing:
        print(col, X_train[col].isna().sum())

    missing = X_test.columns[X_test.isna().any()]       # Test Set
    print(missing)
    for col in missing:
        print(col, X_test[col].isna().sum())

    print("\nTarget Missing")
    print("----------------------")
    print("Train:", y_train.isnull().sum())
    print("Test:", y_test.isnull().sum())


Basic
----------------------
Index([], dtype='str')
Index([], dtype='str')

Target Missing
----------------------
Train: 0
Test: 0

With Property Features
----------------------
Index([], dtype='str')
Index([], dtype='str')

Target Missing
----------------------
Train: 0
Test: 0

With Missing LotSize Flagged
----------------------
Index([], dtype='str')
Index([], dtype='str')

Target Missing
----------------------
Train: 0
Test: 0

With Location Features (City Only)
----------------------
Index([], dtype='str')
Index([], dtype='str')

Target Missing
----------------------
Train: 0
Test: 0

With Location Features (PostalCode Only)
----------------------
Index([], dtype='str')
Index([], dtype='str')

Target Missing
----------------------
Train: 0
Test: 0

With Location Features (County Only)
----------------------
Index([], dtype='str')
Index([], dtype='str')

Target Missing
----------------------
Train: 0
Test: 0

With All Location Features
----------------------
Index([], dtype='str')

In [72]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

results = []

for name, features, in features_sets.items():

    # Define features and target
    X_train = train[features]   # Training Set
    # print(name, X_train.shape)
    y_train = train[target]

    X_test = test[features]     # Test Set
    y_test = test[target]

    # Model
    model = LinearRegression()      # Initialize linear regression model as baseline
    model.fit(X_train, y_train)     # Train model

    # Predictions on target variable
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Compute R^2 scores
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)

    # Save results
    results.append({
        "Feature Set": name,
        "Number of Features": len(features),
        "Training R^2": round(r2_train, 4),
        "Test R^2": round(r2_test, 4)
    })

    '''# Print results
    print(f"{name}")
    print("-----------------------------------")
    print(f"Training R^2: {r2_train:.4f}")
    print(f"Test R^2: {r2_test:.4f}")
    print()'''

In [73]:
results_df = pd.DataFrame(results)
results_df

,Feature Set,Number of Features,Training R^2,Test R^2
0,Basic,4,0.3673,0.3568
1,With Property Features,8,0.3673,0.3568
2,With Missing LotSize Flagged,9,0.3673,0.3568
3,With Location Features (City Only),209,0.3673,0.3568
4,With Location Features (PostalCode Only),209,0.3673,0.3568
5,With Location Features (County Only),68,0.3673,0.3568
6,With All Location Features,470,0.3673,0.3568


Results before removing invalid or extreme values for target (ClosePrice) at the preprocessing step
- Training R^2: 0.0121
- Test R^2: 0.2605

Results after
- Training R^2: 0.3673
- Test R^2: 0.3568

Additional features other than the 4 key variables did not improve the prediction preformances for the linear regression model. So the full feature set ()


Compare top coefficients of the features for training set

In [74]:
from sklearn.preprocessing import StandardScaler

features = features_sets["With All Location Features"]

X_train = train[features]
y_train = train[target]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

model = LinearRegression()
model.fit(X_train_scaled, train[target])

coef_scaled = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_
})

coef_scaled["Abs_Coefficient"] = coef_scaled["Coefficient"].abs()

coef_scaled.sort_values("Abs_Coefficient", ascending=False).head(15)

,Feature,Coefficient,Abs_Coefficient
0,LivingArea,477949.352786,477949.352786
454,CountyOrParish_Santa Clara,329730.602102,329730.602102
447,CountyOrParish_San Bernardino,-192132.272743,192132.272743
444,CountyOrParish_Riverside,-189738.507422,189738.507422
439,CountyOrParish_Orange,170080.714803,170080.714803
163,City_grouped_San Jose,-149594.359085,149594.359085
2,BathroomsTotalInteger,145026.368780,145026.368780
117,City_grouped_Newport Beach,141775.832192,141775.832192
452,CountyOrParish_San Mateo,133253.835264,133253.835264
1,BedroomsTotal,-106990.697935,106990.697935


In [75]:
# Frequency count of each one-hot encoded category
location_groups = {
    "City": city_cols,
    "PostalCode": postal_cols,
    "County": county_cols
}

for name, cols in location_groups.items():
    frequency = train[cols].sum().sort_values(ascending=False)

    print(f"\nTop {name} Frequencies")
    print("--------------------------")
    print(frequency.head(10))


Top City Frequencies
--------------------------
City_grouped_Other          24697.0
City_grouped_Los Angeles     4829.0
City_grouped_San Diego       4012.0
City_grouped_Riverside       1823.0
City_grouped_San Jose        1644.0
City_grouped_Oakland         1506.0
City_grouped_Menifee         1346.0
City_grouped_Long Beach      1231.0
City_grouped_Lancaster       1116.0
City_grouped_Corona          1056.0
dtype: float64

Top PostalCode Frequencies
--------------------------
PostalCode_grouped_Other    60281.0
PostalCode_grouped_92253      840.0
PostalCode_grouped_92345      720.0
PostalCode_grouped_92584      650.0
PostalCode_grouped_92223      620.0
PostalCode_grouped_92592      608.0
PostalCode_grouped_92596      603.0
PostalCode_grouped_92562      562.0
PostalCode_grouped_92211      545.0
PostalCode_grouped_93536      510.0
dtype: float64

Top County Frequencies
--------------------------
CountyOrParish_Los Angeles        29247.0
CountyOrParish_Riverside          17984.0
CountyOrPar

- Another thing to note is that some of the features with relatively large standardized coefficients do not necessarily correspond to the most frequent categories in the dataset. The coefficients here reflects the feature's relationship to ClosePrice.
- LivingArea appears to have the strongest association with predicting ClosePrice.
- Location variables may be meaningful in predicting ClosePrice as well.